# 03 - Baseline Evaluation
- Learning objective: Establish ground-truth BF16 performance numbers (quality, speed, memory) as the reference for quantization experiments.
- Key concepts: HumanEval benchmark, evaluation pipeline, inference performance metrics, validation (81.1% passing per DeepSeek's report)

In [1]:
import torch
import json
import time
import os
import psutil
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

model = AutoModelForCausalLM.from_pretrained("deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct", torch_dtype=torch.bfloat16,)
tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct")

`torch_dtype` is deprecated! Use `dtype` instead!
`rope_parameters`'s factor field must be a float >= 1, got 40
`rope_parameters`'s beta_fast field must be a float, got 32
`rope_parameters`'s beta_slow field must be a float, got 1


Loading weights:   0%|          | 0/351 [00:00<?, ?it/s]

`rope_parameters`'s factor field must be a float >= 1, got 40
`rope_parameters`'s beta_fast field must be a float, got 32
`rope_parameters`'s beta_slow field must be a float, got 1


Now that we've loaded our model, it's time to load and inspect HumanEval:

In [2]:
humaneval = load_dataset("openai/openai_humaneval", split="test")
print(f"Number of problems: {len(humaneval)}")
print(f"Columns: {humaneval.column_names}")

Number of problems: 164
Columns: ['task_id', 'prompt', 'canonical_solution', 'test', 'entry_point']


In [3]:
example = humaneval[0]
for key,value in example.items():
    print(f"\n{'='*40}")
    print(f"{key}:")
    print(f"{'='*40}")
    print(value)


task_id:
HumanEval/0

prompt:
from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    False
    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    True
    """


canonical_solution:
    for idx, elem in enumerate(numbers):
        for idx2, elem2 in enumerate(numbers):
            if idx != idx2:
                distance = abs(elem - elem2)
                if distance < threshold:
                    return True

    return False


test:


METADATA = {
    'author': 'jt',
    'dataset': 'test'
}


def check(candidate):
    assert candidate([1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.3) == True
    assert candidate([1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.05) == False
    assert candidate([1.0, 2.0, 5.9, 4.0, 5.0], 0.95) == True
    assert candidate([1.0, 2.0, 5.9, 4.0, 5.0], 0.8) == Fals

In [4]:
print(tokenizer.chat_template)

{% if not add_generation_prompt is defined %}{% set add_generation_prompt = false %}{% endif %}{{ bos_token }}{% for message in messages %}{% if message['role'] == 'user' %}{{ 'User: ' + message['content'] + '

' }}{% elif message['role'] == 'assistant' %}{{ 'Assistant: ' + message['content'] + eos_token }}{% elif message['role'] == 'system' %}{{ message['content'] + '

' }}{% endif %}{% endfor %}{% if add_generation_prompt %}{{ 'Assistant:' }}{% endif %}


In [5]:
def format_humaneval_prompt(problem):
    instruction = f"Complete the following Python function. Return raw Python code only with correct newlines and indentation. Do not use markdown fences or explanations. You may return either the full function or just the function body.\n\n{problem['prompt']}"

    messages = [{"role": "user", "content": instruction}]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

formatted = format_humaneval_prompt(humaneval[0])
print(formatted)

<｜begin▁of▁sentence｜>User: Complete the following Python function. Return raw Python code only with correct newlines and indentation. Do not use markdown fences or explanations. You may return either the full function or just the function body.

from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    False
    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    True
    """


Assistant:


Test the execution pipeline on this single solution:

### Full Evaluation Loop
Building full evaluation loop with safety and progress tracking

In [6]:
import subprocess
import re
import ast

def execute_with_timeout(code, timeout=10):
    try:
        result = subprocess.run(
            ["python", "-c", code],
            capture_output=True,
            text=True,
            timeout=timeout,
        )
        if result.returncode == 0:
            return "passed"
        stderr = result.stderr.strip()
        if "AssertionError" in stderr:
            return "failed"
        return f"error: {stderr.split(chr(10))[-1]}"
    except subprocess.TimeoutExpired:
        return "error: timeout"
    except Exception as e:
        return f"error: {type(e).__name__}: {e}"
    
def _run_code(code, result_queue):
      try:
          exec_globals = {}
          exec(code, exec_globals)
          result_queue.put("passed")
      except AssertionError:
          result_queue.put("failed")
      except Exception as e:
          result_queue.put(f"error: {type(e).__name__}: {e}")

In [7]:
def generate_completion(problem):
    formatted = format_humaneval_prompt(problem)
    inputs = tokenizer(formatted, return_tensors="pt")

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )
    generated = generated.replace("\r\n", "\n").replace("\r", "\n")
    generated = generated.replace("Ġ", " ").replace("Ċ", "\n").replace("č", "\n")
    return generated.strip()


def has_newlines(text):
    return "\n" in text.strip()


def reconstruct_newlines(code):
    code = re.sub(r'(```python\s*)', '', code)
    code = re.sub(r'(```\s*$)', '', code)
    code = code.strip()

    if has_newlines(code):
        return code

    code = re.sub(
        r'(?<=[^\s])(\s{4,})(?=def |class |return |if |elif |else:|for |while |try:|except |finally:|with |import |from |raise |assert |pass|break|continue)',
        r'\n\1',
        code,
    )
    code = re.sub(r'(?<=:)(    )', r'\n    ', code)

    return code


def strip_code_fences(code):
    code = code.strip()
    blocks = re.findall(r'```(?:python)?\s*\n?(.*?)```', code, re.DOTALL | re.IGNORECASE)
    if blocks:
        code = blocks[0].strip()

    code = re.sub(r'^\s*python(?=from |import |def |class |\n)', '', code, flags=re.IGNORECASE)
    return code.strip()


def reflow_compact_code(code):
    if "\n" in code:
        return code

    # Split glued constructs like "Listdef" or ")def"
    code = re.sub(r'(?<=[A-Za-z0-9_\]\)])(?=(?:def|class)\s)', "\n", code)

    # Split function/class starts if they were flattened
    code = re.sub(r'(?<!^)(?=\b(?:def|class)\b)', "\n", code)

    # Turn inline suite into block form (only when block indent is present)
    code = re.sub(r':(\s{4,})(?=\S)', r':\n\1', code)

    # Fallback: long spacing chunks usually mark missing line breaks
    code = re.sub(r'(?<=[^\s])(\s{4,})(?=\S)', r'\n\1', code)
    return code


def is_valid_python(code):
    try:
        ast.parse(code)
        return True
    except SyntaxError:
        return False


def extract_function_body(candidate, entry_point):
    try:
        tree = ast.parse(candidate)
    except SyntaxError:
        return None

    fn = None
    for node in tree.body:
        if isinstance(node, ast.FunctionDef) and node.name == entry_point:
            fn = node
            break

    if fn is None:
        for node in tree.body:
            if isinstance(node, ast.FunctionDef):
                fn = node
                break

    if fn is None or not fn.body:
        return None

    lines = candidate.splitlines()
    start = fn.body[0].lineno - 1
    end = fn.body[-1].end_lineno
    return "\n".join(lines[start:end]).rstrip()


def salvage_body_from_compact(code):
    compact = code.strip()
    if "def " in compact and ":" in compact:
        compact = compact.split(":", 1)[1].strip()

    if not compact:
        return "    pass"

    parts = [p.strip() for p in re.split(r'\s{4,}', compact) if p.strip()]
    if not parts:
        return "    pass"

    return "\n".join(f"    {p}" for p in parts)


def clean_and_extract(generated, problem):
    base = strip_code_fences(generated)

    candidates = [base]
    if base and not has_newlines(base):
        candidates.append(reconstruct_newlines(base))
        candidates.append(reflow_compact_code(base))

    seen = set()
    for candidate in candidates:
        candidate = candidate.strip()
        if not candidate or candidate in seen:
            continue
        seen.add(candidate)

        if re.match(r'\s*(from |import |class )', candidate) and is_valid_python(candidate):
            return candidate

        if re.match(r'\s*def ', candidate) and is_valid_python(candidate):
            body = extract_function_body(candidate, problem['entry_point'])
            if body:
                stitched = problem['prompt'] + body
                if is_valid_python(stitched):
                    return stitched
            return candidate

        with_prompt = problem["prompt"] + candidate
        if is_valid_python(with_prompt):
            return with_prompt

    if re.match(r'\s*(from |import )', base) and is_valid_python(base):
        return base

    if re.match(r'\s*def ', base) and is_valid_python(base):
        body = extract_function_body(base, problem['entry_point'])
        if body:
            stitched = problem['prompt'] + body
            if is_valid_python(stitched):
                return stitched
        return base

    with_prompt = problem['prompt'] + base
    if is_valid_python(with_prompt):
        return with_prompt

    salvaged = problem['prompt'] + salvage_body_from_compact(base)
    if is_valid_python(salvaged):
        return salvaged

    return with_prompt


def evaluate_problem(problem):
    generated = generate_completion(problem)
    code = clean_and_extract(generated, problem)
    full_code = code + "\n\n" + problem['test'] + f"\ncheck({problem['entry_point']})"
    result = execute_with_timeout(full_code)
    return {"task_id": problem["task_id"], "result": result, "generated": generated}



Test the first 5 problems:

In [8]:
results = []
for i in range(5):
    t0 = time.time()
    r = evaluate_problem(humaneval[i])
    elapsed = time.time() - t0
    results.append(r)
    print(f"{r['task_id']}: {r['result']} ({elapsed:.1f}s)")

passed = sum(1 for r in results if r["result"] == "passed")
print(f"\n{passed}/5 passed")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


HumanEval/0: passed (16.8s)
HumanEval/1: passed (23.6s)
HumanEval/2: passed (5.4s)
HumanEval/3: passed (9.7s)
HumanEval/4: passed (9.4s)

5/5 passed


Run the entire humaneval (164 questions, ~24 minutes)

In [9]:
results = []
t_start = time.time()

for i, problem in enumerate(humaneval):
    t0 = time.time()
    r = evaluate_problem(problem)
    elapsed = time.time() - t0
    r["elapsed_sec"] = elapsed
    results.append(r)

    status = "✓" if r["result"] == "passed" else "✗"
    running_pass = sum(1 for r in results if r["result"] == "passed")
    print(f"{r['task_id']}: {r['result']} ({elapsed:.1f}s)")
    if (i + 1) % 10 == 0 or i == len(humaneval) - 1:
        print(f"[{i+1:3d}/164] {status} {r['task_id']:20s} ({elapsed:.1f}s) | Running: {running_pass}/{i+1} ({100*running_pass/(i+1):.1f}%)")

total_time = time.time() - t_start
passed = sum(1 for r in results if r["result"] == "passed")
print(f"\nFinal: {passed}/164 = {100*passed/164:.1f}% pass@1")
print(f"Total time: {total_time/60:.1f} minutes")

HumanEval/0: passed (8.2s)
HumanEval/1: passed (15.6s)
HumanEval/2: passed (8.4s)
HumanEval/3: passed (7.5s)
HumanEval/4: passed (6.5s)
HumanEval/5: error: NameError: name 'delimiter' is not defined. Did you mean: 'delimeter'? (6.5s)
HumanEval/6: failed (10.7s)
HumanEval/7: passed (6.1s)
HumanEval/8: passed (8.5s)
HumanEval/9: passed (7.2s)
[ 10/164] ✓ HumanEval/9          (7.2s) | Running: 8/10 (80.0%)
HumanEval/10: passed (16.4s)
HumanEval/11: passed (8.8s)
HumanEval/12: passed (9.3s)
HumanEval/13: passed (4.7s)
HumanEval/14: passed (5.3s)
HumanEval/15: passed (4.2s)
HumanEval/16: passed (5.2s)
HumanEval/17: error: timeout (27.3s)
HumanEval/18: passed (8.4s)
HumanEval/19: failed (18.9s)
[ 20/164] ✗ HumanEval/19         (18.9s) | Running: 16/20 (80.0%)
HumanEval/20: passed (14.8s)
HumanEval/21: passed (10.1s)
HumanEval/22: passed (5.6s)
HumanEval/23: passed (2.5s)
HumanEval/24: passed (5.1s)
HumanEval/25: passed (20.4s)
HumanEval/26: failed (8.5s)
HumanEval/27: passed (8.0s)
HumanEval

# Summary
We were able to test the model with HumanEval.  We achieved a benchmark performance of 60.4%.  Below published reference, likely due to prompting/inference/eval setup differences and no pass@k sampling; pipeline now validated and stable for comparative quantization experiments.

Now that we have a baseline performance metric, we are ready to proceed with our quantization study.

In [10]:
import json
import os
import time
from datetime import datetime
from collections import Counter
from statistics import mean, median

# Assumes these exist: results, total_time, passed, model
os.makedirs("../results", exist_ok=True)

n_total = len(results)
n_passed = sum(1 for r in results if r.get("result") == "passed")
n_failed = sum(1 for r in results if r.get("result") == "failed")
n_errors = n_total - n_passed - n_failed
pass_at_1 = (n_passed / n_total * 100) if n_total else 0.0

def parse_error_type(result_str: str) -> str:
    if result_str == "passed":
        return "passed"
    if result_str == "failed":
        return "failed"
    if not isinstance(result_str, str):
        return "unknown"
    if result_str.startswith("error: "):
        tail = result_str[len("error: "):]
        if ":" in tail:
            return tail.split(":", 1)[0].strip()
        return tail.strip()
    return "unknown"

error_type_counts = Counter(parse_error_type(r.get("result", "")) for r in results)

elapsed_sec = [r.get("elapsed_sec") for r in results if isinstance(r.get("elapsed_sec"), (int, float))]
elapsed_sec_sorted = sorted(elapsed_sec)

def percentile(sorted_vals, p):
    if not sorted_vals:
        return None
    idx = int(round((len(sorted_vals) - 1) * p))
    return sorted_vals[idx]

# Prefer measured total_time from eval cell; fallback to sum(elapsed)
_total_time_sec = None
if "total_time" in globals():
    try:
        _total_time_sec = float(total_time)
    except Exception:
        _total_time_sec = None
if _total_time_sec is None and elapsed_sec:
    _total_time_sec = float(sum(elapsed_sec))

runtime_summary = {
    "total_time_sec": _total_time_sec,
    "avg_time_sec_per_problem": mean(elapsed_sec) if elapsed_sec else None,
    "median_time_sec": median(elapsed_sec) if elapsed_sec else None,
    "p95_time_sec": percentile(elapsed_sec_sorted, 0.95),
}

model_id = None
tokenizer_id = None
try:
    model_id = getattr(model, "name_or_path", None)
except Exception:
    pass
try:
    tokenizer_id = getattr(tokenizer, "name_or_path", None)
except Exception:
    pass

generation_config = {
    "max_new_tokens": 512,
    "do_sample": False,
    "skip_special_tokens": True,
    "clean_up_tokenization_spaces": False,
}

stats = {
    "benchmark": "HumanEval",
    "notebook": "03_baseline_evaluation",
    "timestamp_utc": datetime.utcnow().isoformat() + "Z",

    "summary": {
        "n_total": n_total,
        "n_passed": n_passed,
        "n_failed": n_failed,
        "n_errors": n_errors,
        "pass_at_1_percent": pass_at_1,
    },

    "error_type_counts": dict(error_type_counts),
    "runtime": runtime_summary,
    
    "metadata": {
        "model_id": model_id,
        "tokenizer_id": tokenizer_id,
        "hardware": "Apple M4 Max, 64GB RAM",
        "generation_config": generation_config,
    },

    "human_eval_results": results,
}

with open("../results/03_baseline_evaluation.json", "w") as f:
    json.dump(stats, f, indent=2)

/var/folders/1l/ts95kt9945j5dtyz_yqmlmpw0000gn/T/ipykernel_71798/3701335164.py:80: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp_utc": datetime.utcnow().isoformat() + "Z",
